In [1]:
import math
import random
from collections import defaultdict, namedtuple

# define the set of devices and batch multipliers
DEVICES = ["CPU", "GPU", "2xGPU"]
BATCH_MULTIPLIERS = [1.0, 0.5, 2.0, 3.0, 4.0]

# represent a configuration (arm) as a tuple
Arm = namedtuple("Arm", ["device", "batch_mul"])
arms = [Arm(d, b) for d in DEVICES for b in BATCH_MULTIPLIERS]

# UCB‐1 state
counts = defaultdict(int)        # counts[k]: how many times we've tried arm k
values = defaultdict(float)      # values[k]: empirical mean reward of arm k
total_rounds = 0

# SLO deadline and cost‐weight parameters
T_SLO = 1.0       # seconds
alpha = 0.1       # weight for resource cost
beta = 1.0        # penalty weight for SLO violation

def measure_latency_and_cost(arm):
    """
    Stub: run one inference invocation under `arm` configuration,
    return (latency_seconds, mem_cost, gpu_times, gpu_mem_allocs).
    Replace this with your actual profiling harness.
    """
    # e.g. random noise around some hypothetical means
    base = {"CPU": 0.8, "GPU": 0.2, "2xGPU": 0.15}[arm.device]
    latency = random.gauss(base/arm.batch_mul, 0.02)
    mem_cost = 0.5 * latency  # e.g. GB·s
    gpu_times = latency if "GPU" in arm.device else 0.0
    gpu_mem = 1.0 if arm.device=="GPU" else (2.0 if arm.device=="2xGPU" else 0.0)
    return max(latency, 0.0), mem_cost, gpu_times, gpu_mem

def compute_reward(latency, mem_cost, gpu_time, gpu_mem):
    # total resource cost: memory + GPU
    C = mem_cost + gpu_mem * gpu_time
    if latency <= T_SLO:
        reward = (T_SLO - latency) - alpha * C
    else:
        reward = -beta * (latency - T_SLO) - alpha * C
    return reward

def select_arm_ucb():
    global total_rounds
    total_rounds += 1
    # initialize each arm once
    for arm in arms:
        if counts[arm] == 0:
            return arm
    # compute UCB score
    log_t = math.log(total_rounds)
    best_arm, best_score = None, -float("inf")
    for arm in arms:
        mean = values[arm]
        bonus = math.sqrt(2 * log_t / counts[arm])
        score = mean + bonus
        if score > best_score:
            best_score, best_arm = score, arm
    return best_arm

def update_arm(arm, reward):
    counts[arm] += 1
    # incremental mean update
    n = counts[arm]
    values[arm] += (reward - values[arm]) / n

if __name__ == "__main__":
    # run N invocations
    N = 100
    for t in range(N):
        # 1) choose configuration
        arm = select_arm_ucb()
        # 2) execute and measure
        latency, mem_cost, gpu_time, gpu_mem = measure_latency_and_cost(arm)
        # 3) compute reward
        r = compute_reward(latency, mem_cost, gpu_time, gpu_mem)
        # 4) update UCB stats
        update_arm(arm, r)
        print(f"Invocation {t+1:3d}: arm={arm}, latency={latency:.3f}, reward={r:.4f}")

    # after N rounds, show best config
    best = max(arms, key=lambda a: values[a])
    print("\nBest configuration found:", best)


Invocation   1: arm=Arm(device='CPU', batch_mul=1.0), latency=0.820, reward=0.1395
Invocation   2: arm=Arm(device='CPU', batch_mul=0.5), latency=1.609, reward=-0.6895
Invocation   3: arm=Arm(device='CPU', batch_mul=2.0), latency=0.385, reward=0.5955
Invocation   4: arm=Arm(device='CPU', batch_mul=3.0), latency=0.242, reward=0.7459
Invocation   5: arm=Arm(device='CPU', batch_mul=4.0), latency=0.201, reward=0.7886
Invocation   6: arm=Arm(device='GPU', batch_mul=1.0), latency=0.170, reward=0.8046
Invocation   7: arm=Arm(device='GPU', batch_mul=0.5), latency=0.389, reward=0.5525
Invocation   8: arm=Arm(device='GPU', batch_mul=2.0), latency=0.091, reward=0.8952
Invocation   9: arm=Arm(device='GPU', batch_mul=3.0), latency=0.027, reward=0.9693
Invocation  10: arm=Arm(device='GPU', batch_mul=4.0), latency=0.063, reward=0.9272
Invocation  11: arm=Arm(device='2xGPU', batch_mul=1.0), latency=0.143, reward=0.8215
Invocation  12: arm=Arm(device='2xGPU', batch_mul=0.5), latency=0.286, reward=0.6428

In [3]:
import math
import random
from collections import defaultdict, namedtuple

# Define the configuration space
DEVICES = ["CPU", "GPU", "2xGPU"]
BATCH_MULTIPLIERS = [1.0, 0.5, 2.0, 3.0, 4.0]
Arm = namedtuple("Arm", ["device", "batch_mul"])

class MABProfiler:
    """Multi‐Armed Bandit (UCB1) profiler for a single function."""
    def __init__(self, arms):
        self.arms = arms
        self.counts = defaultdict(int)
        self.values = defaultdict(float)
        self.total_rounds = 0

    def select_arm(self):
        """UCB1 selection: ensure each arm is tried once, then use mean+bonus."""
        self.total_rounds += 1
        for arm in self.arms:
            if self.counts[arm] == 0:
                return arm
        log_t = math.log(self.total_rounds)
        best_arm, best_score = None, -float("inf")
        for arm in self.arms:
            mean = self.values[arm]
            bonus = math.sqrt(2 * log_t / self.counts[arm])
            score = mean + bonus
            if score > best_score:
                best_score, best_arm = score, arm
        return best_arm

    def update(self, arm, reward):
        """Update running average for the chosen arm."""
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n

# Simulated measurement and reward functions (to replace with real measurements)
def measure_latency_and_cost(arm):
    base_latency = {"CPU": 0.8, "GPU": 0.2, "2xGPU": 0.15}[arm.device]
    latency = max(random.gauss(base_latency/arm.batch_mul, 0.02), 0.0)
    mem_cost = 0.5 * latency
    gpu_time = latency if "GPU" in arm.device else 0.0
    gpu_mem = 1.0 if arm.device=="GPU" else (2.0 if arm.device=="2xGPU" else 0.0)
    return latency, mem_cost, gpu_time, gpu_mem

T_SLO = 1.0
alpha = 0.1
beta = 1.0
def compute_reward(lat, mem_cost, gpu_time, gpu_mem):
    C = mem_cost + gpu_mem * gpu_time
    if lat <= T_SLO:
        return (T_SLO - lat) - alpha * C
    else:
        return -beta * (lat - T_SLO) - alpha * C

# Manage profilers for multiple functions
profilers = {}  # maps function_id -> MABProfiler

def handle_invocation(function_id, B=30.0):
    """
    Called on each function invocation. Returns chosen (device, batch)
    and simulates execution + profiling update.
    """
    # initialize a profiler for new functions
    if function_id not in profilers:
        arms = [Arm(d, b * B) for d in DEVICES for b in BATCH_MULTIPLIERS]
        profilers[function_id] = MABProfiler(arms)

    prof = profilers[function_id]
    arm = prof.select_arm()
    lat, mem_cost, gpu_time, gpu_mem = measure_latency_and_cost(arm)
    reward = compute_reward(lat, mem_cost, gpu_time, gpu_mem)
    prof.update(arm, reward)
    return arm, lat, reward

# Example usage: simulate 100 invocations of two functions
for t in range(100):
    fn_list = ["video_fn", "image_fn"]
    for func in fn_list:
        arm, latency, rew = handle_invocation(func)
        print(f"[{func}] invocation {t}: arm={arm}, latency={latency:.3f}, reward={rew:.3f}")


[video_fn] invocation 0: arm=Arm(device='CPU', batch_mul=30.0), latency=0.003, reward=0.997
[image_fn] invocation 0: arm=Arm(device='CPU', batch_mul=30.0), latency=0.018, reward=0.981
[video_fn] invocation 1: arm=Arm(device='CPU', batch_mul=15.0), latency=0.032, reward=0.967
[image_fn] invocation 1: arm=Arm(device='CPU', batch_mul=15.0), latency=0.058, reward=0.939
[video_fn] invocation 2: arm=Arm(device='CPU', batch_mul=60.0), latency=0.027, reward=0.972
[image_fn] invocation 2: arm=Arm(device='CPU', batch_mul=60.0), latency=0.005, reward=0.994
[video_fn] invocation 3: arm=Arm(device='CPU', batch_mul=90.0), latency=0.000, reward=1.000
[image_fn] invocation 3: arm=Arm(device='CPU', batch_mul=90.0), latency=0.000, reward=1.000
[video_fn] invocation 4: arm=Arm(device='CPU', batch_mul=120.0), latency=0.000, reward=1.000
[image_fn] invocation 4: arm=Arm(device='CPU', batch_mul=120.0), latency=0.060, reward=0.937
[video_fn] invocation 5: arm=Arm(device='GPU', batch_mul=30.0), latency=0.028,